# Fish Audio S2 Pro — Two-Speaker Travel Dialogue (Kaggle, free T4 x2)

Runs the **open weights** (`fishaudio/s2-pro`) through the official fish-speech CLI on Kaggle's free GPUs. Same steps as `fish_s2_t4_jupyter.ipynb`, with Kaggle paths.

T4 has **no bf16**, so this uses `--half` (fp16). Good for judging pronunciation and which written form works. For a final quality listen use an L4 (`fish_s2_l4_jupyter.ipynb`).

## Before you press Run All

In the right-hand panel (open it with the small arrow at the top right if hidden):

1. **Session options → Accelerator → GPU T4 x2**
2. **Session options → Internet → On** (needs a phone-verified Kaggle account)
3. **Input → Add Input → your dataset** with these five files:
   - the Fish script, e.g. `hindi_fish_bakeoff.txt`
   - its cues, e.g. `hindi_fish_bakeoff.cues.json`
   - `male_ref.wav`, `female_ref.wav`, `ref_texts.json`

Then **Run All**. First run downloads about 11 GB of model weights (a few minutes).

**Where things go:** inputs are copied to `/kaggle/working/fish-s2/inputs`, clips land in `/kaggle/working/fish-s2/outputs/turns`, and the last cell packs everything into `/kaggle/working/fish_clips.zip`.

**Timeline architecture:** Fish generates **one wav per utterance** (same-speaker beat). Inter-turn gaps come from `pause_after_ms` in assemble — not from `<|speaker:N|>` or `[emphasis]`.


In [ ]:
# 1. GPU + internet check (Kaggle T4 — fp16)
import gc
import os
import subprocess
import urllib.request
from pathlib import Path

import torch

assert torch.cuda.is_available() and torch.cuda.device_count() >= 1, (
    "No GPU. Right panel → Session options → Accelerator → GPU T4 x2, then Run All again."
)
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception as e:
    raise SystemExit(
        "No internet. Right panel → Session options → Internet → On "
        f"(needs a phone-verified account). Details: {e}"
    )

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ROOT = Path("/kaggle/working/fish-s2")
INPUT_DIR = ROOT / "inputs"
OUT_DIR = ROOT / "outputs"
FISH_REPO = Path("/tmp/fish-s2/fish-speech")
for d in (ROOT, INPUT_DIR, OUT_DIR, FISH_REPO.parent):
    d.mkdir(parents=True, exist_ok=True)

gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

gc.collect()
torch.cuda.empty_cache()

try:
    print("nvidia-smi:", subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
        text=True,
    ).strip())
except Exception as e:
    print("nvidia-smi:", e)

print(f"GPUs: {torch.cuda.device_count()} | GPU 0: {gpu_name} | VRAM {vram_gb:.1f} GB | capability={capability}")
print(f"torch {torch.__version__} | ROOT={ROOT}")
print("This notebook: --half (fp16), one Fish call per utterance, --no-compile, GPU 0 only")


In [ ]:
# 2. Copy inputs from the attached Kaggle dataset into the working folder
import shutil
from pathlib import Path

# Leave "" to use the first *fish*.txt found. Set a file name to choose, e.g. "chokhi_dhani_fish_hi.txt".
SCRIPT_NAME = ""

ROOT = Path("/kaggle/working/fish-s2")
INPUT_DIR = ROOT / "inputs"
OUT_DIR = ROOT / "outputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_INPUT = Path("/kaggle/input")
wanted_suffixes = (".txt", ".json", ".wav")
found = [p for p in KAGGLE_INPUT.rglob("*") if p.is_file() and p.suffix.lower() in wanted_suffixes]
for src in found:
    shutil.copy2(src, INPUT_DIR / src.name)
    print("copied:", src)

scripts = sorted(INPUT_DIR.glob("*.txt"))
assert scripts, (
    f"No script .txt found under {KAGGLE_INPUT}. "
    "Right panel → Input → Add Input → pick your dataset, then Run All again."
)
if SCRIPT_NAME:
    SCRIPT_FILE = INPUT_DIR / SCRIPT_NAME
    assert SCRIPT_FILE.exists(), f"{SCRIPT_NAME} not in {sorted(p.name for p in scripts)}"
else:
    fish_scripts = sorted(INPUT_DIR.glob("*fish*.txt"))
    SCRIPT_FILE = (fish_scripts or scripts)[0]
have_refs = (INPUT_DIR / "male_ref.wav").exists() and (INPUT_DIR / "female_ref.wav").exists()
assert have_refs, (
    f"Missing male_ref.wav / female_ref.wav. Found: {sorted(p.name for p in INPUT_DIR.iterdir())}. "
    "Add them to the dataset."
)
print(f"Script: {SCRIPT_FILE.name}")
print(f"Cues: {SCRIPT_FILE.with_suffix('.cues.json').exists()}")
print("Refs: OK")
print(f"ref_texts.json: {(INPUT_DIR / 'ref_texts.json').exists()}")
print("inputs:", sorted(p.name for p in INPUT_DIR.iterdir()))


## Check refs
Listen before spending generate time. Refs set **timbre / accent / baseline energy**. Transcripts in `ref_texts.json` must match the wav words.

In [ ]:
# 3. Listen to refs
import IPython.display as ipd
for name in ("male_ref.wav", "female_ref.wav"):
    p = INPUT_DIR / name
    assert p.exists(), p
    print(name)
    display(ipd.Audio(str(p)))

In [ ]:
# 4. Install fish-speech deps (official CLI, not SGLang, not paid API)
# Adapted from fish_s2_t4_jupyter.ipynb. Do NOT `pip install -e .` the repo.
# Keep Kaggle's torch. Never install torchcodec.
import re
import site
import subprocess
import sys
from pathlib import Path

import torch

ROOT = Path("/kaggle/working/fish-s2")
FISH_REPO = Path("/tmp/fish-s2/fish-speech")
ROOT.mkdir(parents=True, exist_ok=True)
FISH_REPO.parent.mkdir(parents=True, exist_ok=True)

if not (FISH_REPO / ".git").is_dir():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/fishaudio/fish-speech.git",
        str(FISH_REPO),
    ])
else:
    print("repo already cloned:", FISH_REPO)

user_site = Path(site.getusersitepackages())
user_site.mkdir(parents=True, exist_ok=True)
(user_site / "fish-speech.pth").write_text(str(FISH_REPO) + "\n")
print("PYTHONPATH via", user_site / "fish-speech.pth")

PIP = [sys.executable, "-m", "pip", "install", "--user"]
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "hydra-core", "hydra"])
subprocess.check_call(PIP + ["--no-cache-dir", "hydra-core==1.3.2", "omegaconf==2.3.0"])


def _pip_pkg(spec: str, *, only_binary: bool = True) -> None:
    print("installing", spec, flush=True)
    cmd = PIP + (["--only-binary=:all:", spec] if only_binary else [spec])
    if subprocess.call(cmd) != 0:
        if only_binary:
            print("  retry without only-binary:", spec, flush=True)
            subprocess.check_call(PIP + [spec])
        else:
            raise SystemExit(f"pip failed: {spec}")


for spec in (
    "click", "loguru", "einops", "natsort", "safetensors", "tqdm",
    "tiktoken", "pydantic>=2", "lightning", "librosa", "soundfile", "pydub",
    "loralib", "huggingface_hub", "einx", "transformers",
    "protobuf>=3.20,<6", "resampy", "ormsgpack", "cachetools",
):
    _pip_pkg(spec)

for spec in ("descript-audiotools", "descript-audio-codec"):
    _pip_pkg(spec)

# Match torchaudio to Kaggle's torch; do not pip-install torch.
ta_ver = torch.__version__.split("+")[0]
try:
    import torchaudio
    have_ta = torchaudio.__version__.split("+")[0]
except Exception:
    have_ta = None
if have_ta == ta_ver:
    print(f"torchaudio {have_ta} already matches torch {torch.__version__}")
else:
    cuda = torch.version.cuda or "12.4"
    cu_tag = "cu" + cuda.replace(".", "")
    print(f"torch {torch.__version__} cuda {cuda} → torchaudio=={ta_ver} ({cu_tag}); had {have_ta}")
    subprocess.check_call(
        PIP + ["--no-deps", f"torchaudio=={ta_ver}",
               "--index-url", f"https://download.pytorch.org/whl/{cu_tag}"]
    )

# torchaudio 2.9 load() needs torchcodec (CUDA 13). Use soundfile instead.
infer_py = FISH_REPO / "fish_speech" / "models" / "text2semantic" / "inference.py"
src = infer_py.read_text(encoding="utf-8")
pat = re.compile(r"^(\s*)wav, sr = torchaudio\.load\(str\(audio_path\)\)\s*$", re.M)
if "sf.read(str(audio_path)" in src:
    print("encode_audio already uses soundfile")
else:
    m = pat.search(src)
    assert m, f"could not find torchaudio.load in {infer_py}"
    ind = m.group(1)
    repl = (
        f"{ind}import soundfile as sf\n"
        f"{ind}data, sr = sf.read(str(audio_path), dtype=\"float32\", always_2d=True)\n"
        f"{ind}wav = torch.from_numpy(data.T.copy())  # (channels, samples)"
    )
    infer_py.write_text(pat.sub(repl, src, count=1), encoding="utf-8")
    print("patched encode_audio → soundfile")

subprocess.check_call([
    sys.executable, "-c",
    "import hydra; assert hydra.__version__.startswith('1.3'), hydra.__version__; "
    "print('hydra', hydra.__version__)",
])
print("Installed. Repo:", FISH_REPO)
# T4 memory: by default Fish reserves room for 32768 tokens (about 6 GB of cache + mask),
# which leaves no space for the codec on a 15 GB card. One utterance needs far less.
MAX_SEQ_LEN = 4096
load_old = "DualARTransformer.from_pretrained(checkpoint_path, load_weights=True)"
load_new = f"DualARTransformer.from_pretrained(checkpoint_path, load_weights=True, max_length={MAX_SEQ_LEN})"
src = infer_py.read_text(encoding="utf-8")
if "load_weights=True, max_length=" in src:
    print("max_seq_len already capped")
else:
    assert load_old in src, f"could not find model load call in {infer_py}"
    infer_py.write_text(src.replace(load_old, load_new, 1), encoding="utf-8")
    print(f"patched model load → max_seq_len {MAX_SEQ_LEN}")


In [ ]:
# 5. Prefetch S2 Pro open weights (avoids mid-run downloads)
from huggingface_hub import snapshot_download
from pathlib import Path

ROOT = Path("/kaggle/working/fish-s2")
FISH_REPO = Path("/tmp/fish-s2/fish-speech")
CKPT_DIR = FISH_REPO / "checkpoints" / "s2-pro"
MODEL_ID = "fishaudio/s2-pro"

CKPT_DIR.mkdir(parents=True, exist_ok=True)
if any(CKPT_DIR.rglob("codec.pth")):
    print("Weights already present — skip download")
else:
    print("Downloading", MODEL_ID, "→", CKPT_DIR)
    snapshot_download(MODEL_ID, local_dir=str(CKPT_DIR))
assert (CKPT_DIR / "codec.pth").exists() or any(CKPT_DIR.rglob("codec.pth")), (
    f"No codec.pth under {CKPT_DIR} — check the snapshot layout"
)
print("Weights cached. codec.pth:", next(CKPT_DIR.rglob("codec.pth"), CKPT_DIR / "codec.pth"))

In [ ]:
# 6. Config — T4 knobs (fp16). Utterance timeline is in the parse cell.
import json
from pathlib import Path

ROOT = Path("/kaggle/working/fish-s2")
INPUT_DIR = ROOT / "inputs"
OUT_DIR = ROOT / "outputs"
FISH_REPO = Path("/tmp/fish-s2/fish-speech")
CKPT_DIR = FISH_REPO / "checkpoints" / "s2-pro"
_codec = next(CKPT_DIR.rglob("codec.pth"), None)
assert _codec, f"No codec.pth under {CKPT_DIR} — re-run the weights cell"
CKPT_DIR = _codec.parent  # from_pretrained wants the dir that contains codec.pth
SPEAKER0_REF = INPUT_DIR / "male_ref.wav"
SPEAKER1_REF = INPUT_DIR / "female_ref.wav"
assert "SCRIPT_FILE" in dir(), "Run the inputs cell (2) first"

ref_texts_path = INPUT_DIR / "ref_texts.json"
if ref_texts_path.exists():
    _texts = json.loads(ref_texts_path.read_text(encoding="utf-8"))
    REF_TEXT_M, REF_TEXT_F = _texts["male"], _texts["female"]
    print("Ref transcripts from ref_texts.json")
else:
    REF_TEXT_M = (
        "You know, this place has a real pull to it. I've been around a few spots "
        "like this, but here at Chokhee Dhaanee you can feel why people stay. "
        "Let me show you something interesting — take it at an easy pace with me."
    )
    REF_TEXT_F = (
        "Wait — that's actually fascinating. So this isn't only for show? "
        "I love how that changes how you look at Chokhee Dhaanee. "
        "Hah, okay, tell me about the Kalbeliya dancers next. I'm genuinely curious."
    )
    print("Using default ref transcripts — edit if your wav words differ.")

# T4-ONLY knobs (per-utterance generate; TURNS_PER_CHUNK unused)
TURNS_PER_CHUNK = 0
CHUNK_LENGTH = 240
USE_HALF = True
USE_COMPILE = False
SMOKE_TURNS = 6
SEED = 42

assert SPEAKER0_REF.exists() and SPEAKER1_REF.exists()
assert CKPT_DIR.is_dir(), CKPT_DIR
print(f"Script: {SCRIPT_FILE.name}")
print(f"S0={SPEAKER0_REF.name} | S1={SPEAKER1_REF.name}")
print(
    f"TURNS_PER_CHUNK={TURNS_PER_CHUNK} CHUNK_LENGTH={CHUNK_LENGTH} "
    f"USE_HALF={USE_HALF} USE_COMPILE={USE_COMPILE}"
)


In [ ]:
# 7. Parse directed script → utterances (same-speaker beats) + timeline
# Fish speaker tokens do NOT create turn gaps. We generate per utterance, then
# insert pause_after_ms silence in the assemble cell.
import json
import re
from pathlib import Path

SPEAKER_LINE = re.compile(
    r"^\s*Speaker\s+(?P<num>1|2)\s*:\s*(?P<text>.+?)\s*$", re.IGNORECASE
)
TAG_LINE = re.compile(
    r"^\s*\[(?P<tag>M|F)\]\s*(?P<text>.+?)\s*$", re.IGNORECASE
)
NUM_TO_FISH = {"1": "0", "2": "1"}
TAG_TO_FISH = {"M": "0", "F": "1"}

MAX_UTT_WORDS = 55
MAX_UTT_TURNS = 3
HOLD_PATTERNS = [
    re.compile(p, re.I)
    for p in (
        r"take a (quiet )?second",
        r"take a moment",
        r"look around",
        r"if you can see",
        r"if you hear",
        r"listen for",
        r"notice ",
        r"soak (it|this) in",
    )
]


def parse_turns(text: str):
    turns = []
    for line in text.splitlines():
        if not line.strip():
            continue
        if m := SPEAKER_LINE.match(line):
            turns.append((NUM_TO_FISH[m.group("num")], m.group("text").strip()))
        elif m := TAG_LINE.match(line):
            turns.append((TAG_TO_FISH[m.group("tag").upper()], m.group("text").strip()))
    return turns


def strip_tags(text: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"\[[^\]]+\]", "", text)).strip()


def pause_after_ms(text: str, speaker: str, next_speaker, index: int) -> int:
    spoken = strip_tags(text)
    change = next_speaker is not None and next_speaker != speaker
    if spoken.endswith("?"):
        base = 520 if change else 380
    elif spoken.endswith("!"):
        base = 280 if change else 200
    elif "…" in text or "..." in text or "—" in text:
        base = 420 if change else 300
    elif change:
        base = 360 + (index % 4) * 40
    else:
        base = 160 + (index % 3) * 40
    hold = 2200 if any(p.search(spoken) for p in HOLD_PATTERNS) else 0
    return max(120, min(2500, max(base, hold)))


def load_or_build_cues(turns):
    # Cues that belong to the chosen script first; any cues file otherwise.
    cues_path = SCRIPT_FILE.with_suffix(".cues.json")
    if not cues_path.exists():
        cues_path = next(iter(sorted(INPUT_DIR.glob("*.cues.json"))), None)
    if cues_path and cues_path.exists():
        raw = json.loads(cues_path.read_text(encoding="utf-8"))
        print(f"Loaded cues: {cues_path.name} ({len(raw)} lines)")
        cues = []
        for i, c in enumerate(raw):
            row = dict(c)
            sp = str(row.get("speaker") or "M").upper()
            row["speaker"] = "M" if sp in ("M", "1") else "F"
            if "tagged_text" not in row:
                row["tagged_text"] = row.get("text") or ""
            if "pause_after_ms" not in row or not row["pause_after_ms"]:
                nxt = None
                if i + 1 < len(raw):
                    ns = str(raw[i + 1].get("speaker") or "").upper()
                    nxt = "M" if ns in ("M", "1") else "F"
                row["pause_after_ms"] = pause_after_ms(
                    row.get("tagged_text") or row.get("text") or "",
                    row["speaker"],
                    nxt,
                    i,
                )
            cues.append(row)
        return cues

    print("No cues.json in inputs — computing pause_after_ms on VM")
    cues = []
    for i, (sid, text) in enumerate(turns):
        speaker = "M" if sid == "0" else "F"
        nxt = None
        if i + 1 < len(turns):
            nxt = "M" if turns[i + 1][0] == "0" else "F"
        pa = pause_after_ms(text, speaker, nxt, i)
        cues.append(
            {
                "index": i + 1,
                "speaker": speaker,
                "text": strip_tags(text),
                "tagged_text": text,
                "section": "culture",
                "pause_after_ms": pa,
                "hold_after_ms": 2200 if pa >= 1800 else 0,
            }
        )
    if cues:
        cues[-1]["pause_after_ms"] = max(int(cues[-1]["pause_after_ms"]), 600)
    return cues


def group_utterances(cues):
    groups = []
    buf = []

    def fish_sid(speaker: str) -> str:
        return "0" if speaker == "M" else "1"

    def flush():
        nonlocal buf
        if not buf:
            return
        speaker = buf[0]["speaker"]
        tagged = [c["tagged_text"] for c in buf]
        sid = fish_sid(speaker)
        fish_text = "\n".join(f"<|speaker:{sid}|> {t}" for t in tagged)
        pause = max(int(c.get("pause_after_ms") or 0) for c in buf)
        groups.append(
            {
                "utterance_index": len(groups) + 1,
                "speaker": speaker,
                "section": buf[0].get("section") or "culture",
                "cue_indexes": [c.get("index") for c in buf],
                "fish_text": fish_text,
                "pause_after_ms": pause,
                "words": sum(len(str(c.get("text") or "").split()) for c in buf),
            }
        )
        buf = []

    for c in cues:
        if not buf:
            buf = [c]
            continue
        same = c["speaker"] == buf[0]["speaker"]
        words = sum(len(str(x.get("text") or "").split()) for x in buf) + len(
            str(c.get("text") or "").split()
        )
        if same and len(buf) < MAX_UTT_TURNS and words <= MAX_UTT_WORDS:
            buf.append(c)
        else:
            flush()
            buf = [c]
    flush()
    return groups


turns = parse_turns(SCRIPT_FILE.read_text(encoding="utf-8"))
assert turns, "No Speaker 1/2 or [M]/[F] lines found"
CUES = load_or_build_cues(turns)
UTTERANCES = group_utterances(CUES)
CLIPS_DIR = OUT_DIR / "turns"
CLIPS_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "utterances.json").write_text(
    json.dumps(UTTERANCES, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)

# Legacy names (smoke / optional monolith) — smoke uses first few utterances
FISH_TEXT = "\n".join(u["fish_text"] for u in UTTERANCES)
FISH_SMOKE_TEXT = "\n".join(u["fish_text"] for u in UTTERANCES[: max(1, min(2, len(UTTERANCES)))])
# Keep FISH_CHUNKS for T4 old path compatibility = one chunk per utterance
FISH_CHUNKS = [u["fish_text"] for u in UTTERANCES]

gaps = [int(u["pause_after_ms"]) for u in UTTERANCES]
print(
    f"{len(turns)} script turns → {len(UTTERANCES)} utterances, "
    f"{sum(u['words'] for u in UTTERANCES)} words"
)
print(
    f"pause_after_ms min/avg/max = "
    f"{min(gaps)}/{sum(gaps)//len(gaps)}/{max(gaps)}"
)
print(f"Clips dir: {CLIPS_DIR}")
print("Preview utterance 1:")
print(UTTERANCES[0]["fish_text"][:300])


In [ ]:
# 8. CLI helper + smoke generate (first utterance). T4: fp16 via USE_HALF.
import gc
import os
import site
import subprocess
import sys
from pathlib import Path

import torch
import IPython.display as ipd

assert torch.cuda.is_available(), "GPU lost — check Session options → Accelerator"
gc.collect()
torch.cuda.empty_cache()
print("VRAM before generate:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB allocated")

OUT_DIR.mkdir(parents=True, exist_ok=True)
CLIPS_DIR = OUT_DIR / "turns"
CLIPS_DIR.mkdir(parents=True, exist_ok=True)
INFER = FISH_REPO / "fish_speech" / "models" / "text2semantic" / "inference.py"
assert INFER.exists(), INFER


def run_fish_inference(text: str, out_wav: Path) -> None:
    cmd = [
        sys.executable,
        str(INFER),
        "--text", text,
        "--prompt-audio", str(SPEAKER0_REF),
        "--prompt-text", REF_TEXT_M,
        "--prompt-audio", str(SPEAKER1_REF),
        "--prompt-text", REF_TEXT_F,
        "--checkpoint-path", str(CKPT_DIR),
        "--output", str(out_wav),
        "--output-dir", str(OUT_DIR / "codes"),
        "--device", "cuda",
        "--seed", str(SEED),
        "--chunk-length", str(CHUNK_LENGTH),
    ]
    cmd.append("--compile" if USE_COMPILE else "--no-compile")
    if USE_HALF:
        cmd.append("--half")
    env = os.environ.copy()
    env["PYTHONPATH"] = os.pathsep.join([
        site.getusersitepackages(),
        str(FISH_REPO),
        env.get("PYTHONPATH", ""),
    ])
    print("Running:", " ".join(cmd[:6]), "... →", out_wav.name)
    proc = subprocess.run(
        cmd,
        cwd=str(FISH_REPO),
        env=env,
        capture_output=True,
        text=True,
    )
    if proc.stdout:
        print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print(proc.stderr[-6000:] if proc.stderr else "(no stderr)")
        raise RuntimeError(f"inference.py failed ({proc.returncode}) for {out_wav.name}")
    assert out_wav.exists() and out_wav.stat().st_size > 1000, out_wav
    print(f"  -> {out_wav} ({out_wav.stat().st_size / 1e6:.1f} MB)")


def utt_path(u: dict) -> Path:
    sp = "M" if u["speaker"] == "M" else "F"
    return CLIPS_DIR / f"utt_{int(u['utterance_index']):03d}_{sp}.wav"


# Smoke: generate first smoke utterance only (timeline path)
smoke_u = UTTERANCES[0]
SMOKE_PATH = utt_path(smoke_u)
print(f"=== smoke utterance 1/{len(UTTERANCES)} ({smoke_u['words']} words) ===")
run_fish_inference(smoke_u["fish_text"], SMOKE_PATH)
ipd.Audio(str(SMOKE_PATH))


In [ ]:
# 9. Full episode: one Fish call per utterance → turns/*.wav
# Timeline architecture: Fish = voice/prosody inside a beat; gaps come later.
import gc

import torch

assert "run_fish_inference" in dir(), "Run the smoke generate cell first (defines the CLI helper)."
assert UTTERANCES, "Run the parse cell first."
gc.collect()
torch.cuda.empty_cache()

CLIPS_DIR = OUT_DIR / "turns"
CLIPS_DIR.mkdir(parents=True, exist_ok=True)

for u in UTTERANCES:
    out_path = utt_path(u)
    if out_path.exists() and out_path.stat().st_size > 1000:
        print(f"skip existing {out_path.name}")
        continue
    print(
        f"\n=== utterance {u['utterance_index']}/{len(UTTERANCES)} "
        f"{u['speaker']} ({u['words']}w, gap={u['pause_after_ms']}ms) ==="
    )
    run_fish_inference(u["fish_text"], out_path)
    gc.collect()
    torch.cuda.empty_cache()

print("All utterance clips:", sorted(p.name for p in CLIPS_DIR.glob("utt_*.wav")))


In [ ]:
# 10. Assemble episode with deterministic pause_after_ms gaps
from pathlib import Path

import IPython.display as ipd
from pydub import AudioSegment

OUT_DIR = Path("/kaggle/working/fish-s2/outputs")
CLIPS_DIR = OUT_DIR / "turns"
assert UTTERANCES, "Run parse cell first"
clips = sorted(CLIPS_DIR.glob("utt_*.wav"))
assert clips, f"No utt_*.wav in {CLIPS_DIR} — run full generate first"
print("Found clips:", [c.name for c in clips])


def trim_silence(seg: AudioSegment, silence_thresh_db: int = -40) -> AudioSegment:
    # light edge trim so gaps are the intentional ones
    try:
        from pydub.silence import detect_leading_silence
        start_trim = detect_leading_silence(seg, silence_threshold=silence_thresh_db)
        end_trim = detect_leading_silence(seg.reverse(), silence_threshold=silence_thresh_db)
        start_trim = min(start_trim, len(seg) // 4)
        end_trim = min(end_trim, len(seg) // 4)
        return seg[start_trim : len(seg) - end_trim]
    except Exception:
        return seg


episode = AudioSegment.empty()
for u in UTTERANCES:
    sp = "M" if u["speaker"] == "M" else "F"
    path = CLIPS_DIR / f"utt_{int(u['utterance_index']):03d}_{sp}.wav"
    assert path.exists(), path
    clip = trim_silence(AudioSegment.from_wav(path))
    episode += clip
    gap = int(u.get("pause_after_ms") or 0)
    if gap > 0:
        episode += AudioSegment.silent(duration=gap)

EPISODE_PATH = OUT_DIR / "episode.wav"
episode.export(EPISODE_PATH, format="wav")
print(
    f"{len(UTTERANCES)} utterances → {EPISODE_PATH} "
    f"({len(episode) / 60000:.1f} min) with explicit turn gaps"
)
ipd.Audio(str(EPISODE_PATH))


In [ ]:
# 11. Listen here (grouped by written form when the cues carry one) + pack a zip to download
import os
import shutil

import IPython.display as ipd
from IPython.display import FileLink

form_by_index = {int(c.get("index") or 0): c.get("form") for c in CUES}
current = None
for u in UTTERANCES:
    first = (u.get("cue_indexes") or [0])[0]
    form = form_by_index.get(int(first or 0)) or "episode"
    if form != current:
        print("\n" + "=" * 24, form, "=" * 24)
        current = form
    p = utt_path(u)
    print(p.name, "|", u["fish_text"].replace("\n", " ")[:160])
    display(ipd.Audio(str(p)))

os.chdir("/kaggle/working")
zip_path = shutil.make_archive("/kaggle/working/fish_clips", "zip", str(OUT_DIR))
print("\nAll clips + episode.wav packed:", zip_path)
print("Download: right panel → Output → fish_clips.zip → ⋮ → Download, or click the link below.")
display(FileLink("fish_clips.zip"))


## Troubleshooting (S2 Pro on Kaggle)

- **"No GPU"** — right panel → Session options → Accelerator → **GPU T4 x2**. Changing it restarts the session; press Run All again.
- **"No internet"** — Session options → Internet → On. Kaggle only shows this switch after phone verification (Settings → Phone verification).
- **"No script .txt found"** — the dataset is not attached. Right panel → Input → Add Input → Your Datasets.
- **Install cell fails** — copy the last 30 lines of the red text. Do not `pip install -e .` the repo.
- **CUDA out of memory** — S2 Pro in fp16 is close to a T4's 15 GB. The install cell caps `MAX_SEQ_LEN` at 4096 to make room; if it still happens, lower it to 2048 in that cell and Run All again.
- **Stops half way** — Run All again. Finished clips are skipped, so it resumes.
- **Quality** — T4 fp16 is a notch below L4 bf16. Judge pronunciation and written form here; judge final polish on an L4.
- **Weekly quota** — Kaggle shows remaining GPU hours under your profile picture. Stop the session (power icon) when done so it does not burn quota.
- **Back on the Mac** — unzip `fish_clips.zip`; `turns/` holds one wav per utterance, `episode.wav` is already assembled with gaps. Ambience: `python run_mix.py --episode episode.wav --cues <cues.json> --no-holds`.
